In [1]:
# Melbourne Housing - Data Preprocessing & Feature Engineering
# Dataset: https://www.kaggle.com/datasets/anthonypino/melbourne-housing-market

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

df = pd.read_csv(r"C:\Users\syin\Downloads\archive (33)\Melbourne_housing_FULL.csv")



In [2]:
# ---- 1. First look ----
print(df.shape)
print(df.info())


(34857, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34857 entries, 0 to 34856
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         34857 non-null  object 
 1   Address        34857 non-null  object 
 2   Rooms          34857 non-null  int64  
 3   Type           34857 non-null  object 
 4   Price          27247 non-null  float64
 5   Method         34857 non-null  object 
 6   SellerG        34857 non-null  object 
 7   Date           34857 non-null  object 
 8   Distance       34856 non-null  float64
 9   Postcode       34856 non-null  float64
 10  Bedroom2       26640 non-null  float64
 11  Bathroom       26631 non-null  float64
 12  Car            26129 non-null  float64
 13  Landsize       23047 non-null  float64
 14  BuildingArea   13742 non-null  float64
 15  YearBuilt      15551 non-null  float64
 16  CouncilArea    34854 non-null  object 
 17  Lattitude      26881 non-null  float64

In [3]:
# ---- 2. Check missing values ----
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

# Melbourne housing usually has a lot of missing values in:
# BuildingArea, YearBuilt, Car, CouncilArea, Landsize


BuildingArea     21115
YearBuilt        19306
Landsize         11810
Car               8728
Bathroom          8226
Bedroom2          8217
Longtitude        7976
Lattitude         7976
Price             7610
CouncilArea          3
Regionname           3
Propertycount        3
Postcode             1
Distance             1
dtype: int64


In [4]:
# ---- 3. Drop columns with too much missing data ----
# BuildingArea and YearBuilt are often >40-50% missing - not worth imputing heavily
df = df.drop(columns=["BuildingArea", "YearBuilt"])



In [5]:
# ---- 4. Handle remaining missing values ----

# Car (number of car spots) - numeric, fill with median
df["Car"] = df["Car"].fillna(df["Car"].median())

# Landsize - numeric, fill with median (has outliers/zeros too, median is safer than mean)
df["Landsize"] = df["Landsize"].fillna(df["Landsize"].median())

# CouncilArea - categorical, fill with mode
df["CouncilArea"] = df["CouncilArea"].fillna(df["CouncilArea"].mode()[0])

# Price is the target - can't predict/train on rows with missing target, so drop those rows
df = df.dropna(subset=["Price"])

print(df.isnull().sum().sum(), "missing values remaining")


25404 missing values remaining


In [6]:
# ---- 5. Drop columns that aren't useful for modelling ----
# Address is free text/unique per row, Date isn't needed in raw form yet
df = df.drop(columns=["Address"])

# ---- 6. Feature engineering from Date ----
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df["SaleYear"] = df["Date"].dt.year
df["SaleMonth"] = df["Date"].dt.month
df = df.drop(columns=["Date"])

# ---- 7. Property age at time of sale would've been nice, but we dropped YearBuilt
#          so skipping that here - just noting this is a common feature to add
#          when YearBuilt isn't heavily missing


In [7]:
# ---- 8. Categorical encoding ----

# Type (h = house, u = unit, t = townhouse) - few categories, no order -> one-hot
df = pd.get_dummies(df, columns=["Type"], drop_first=True)

# Method (sale method: S, SP, PI, VB, etc.) - one-hot as well
df = pd.get_dummies(df, columns=["Method"], drop_first=True)

# Regionname - one-hot
df = pd.get_dummies(df, columns=["Regionname"], drop_first=True)

# Suburb and CouncilArea have too many unique categories for one-hot
# (would create hundreds of columns) - use label/frequency encoding instead
suburb_counts = df["Suburb"].value_counts()
df["Suburb_freq"] = df["Suburb"].map(suburb_counts)
df = df.drop(columns=["Suburb"])

council_counts = df["CouncilArea"].value_counts()
df["CouncilArea_freq"] = df["CouncilArea"].map(council_counts)
df = df.drop(columns=["CouncilArea"])

# SellerG has many unique agents too - same approach
sellerg_counts = df["SellerG"].value_counts()
df["SellerG_freq"] = df["SellerG"].map(sellerg_counts)
df = df.drop(columns=["SellerG"])

print(df.head())
print(df.shape)


   Rooms      Price  Distance  Postcode  Bedroom2  Bathroom  Car  Landsize  \
1      2  1480000.0       2.5    3067.0       2.0       1.0  1.0     202.0   
2      2  1035000.0       2.5    3067.0       2.0       1.0  0.0     156.0   
4      3  1465000.0       2.5    3067.0       3.0       2.0  0.0     134.0   
5      3   850000.0       2.5    3067.0       3.0       2.0  1.0      94.0   
6      4  1600000.0       2.5    3067.0       3.0       1.0  2.0     120.0   

   Lattitude  Longtitude  ...  Regionname_Eastern Victoria  \
1   -37.7996    144.9984  ...                        False   
2   -37.8079    144.9934  ...                        False   
4   -37.8093    144.9944  ...                        False   
5   -37.7969    144.9969  ...                        False   
6   -37.8072    144.9941  ...                        False   

   Regionname_Northern Metropolitan  Regionname_Northern Victoria  \
1                              True                         False   
2                   

In [8]:
# ---- 9. Outlier check on Price (target) ----
print(df["Price"].describe())

# cap extreme outliers using IQR method instead of dropping (keeps row count)
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df["Price"] = df["Price"].clip(lower=lower, upper=upper)



count    2.724700e+04
mean     1.050173e+06
std      6.414671e+05
min      8.500000e+04
25%      6.350000e+05
50%      8.700000e+05
75%      1.295000e+06
max      1.120000e+07
Name: Price, dtype: float64


In [9]:
# ---- 10. Scale numeric features ----
numeric_cols = ["Rooms", "Distance", "Landsize", "Car", "Lattitude", "Longtitude"]
numeric_cols = [c for c in numeric_cols if c in df.columns]

scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print(df.head())




      Rooms      Price  Distance  Postcode  Bedroom2  Bathroom       Car  \
1 -1.039292  1480000.0 -1.293625    3067.0       2.0       1.0 -0.904694   
2 -1.039292  1035000.0 -1.293625    3067.0       2.0       1.0 -2.054744   
4  0.008072  1465000.0 -1.293625    3067.0       3.0       2.0 -2.054744   
5  0.008072   850000.0 -1.293625    3067.0       3.0       2.0 -0.904694   
6  1.055436  1600000.0 -1.293625    3067.0       3.0       1.0  0.245357   

   Landsize  Lattitude  Longtitude  ...  Regionname_Eastern Victoria  \
1 -0.120179   0.080367    0.013996  ...                        False   
2 -0.135249  -0.010227   -0.027437  ...                        False   
4 -0.142457  -0.025508   -0.019150  ...                        False   
5 -0.155561   0.109838    0.001566  ...                        False   
6 -0.147043  -0.002587   -0.021636  ...                        False   

   Regionname_Northern Metropolitan  Regionname_Northern Victoria  \
1                              True      

In [10]:
# ---- 11. Final check ----
print(df.shape)
print(df.isnull().sum().sum(), "missing values remaining")

# ---- 12. Save cleaned dataset ----
df.to_csv("melbourne_housing_processed.csv", index=False)
print("Saved -> melbourne_housing_processed.csv")


(27247, 29)
25401 missing values remaining
Saved -> melbourne_housing_processed.csv
